<a href="https://colab.research.google.com/github/alauraura/FedMob-HAR/blob/main/PAMAP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Pré-processamento

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from google.colab import drive
import os

# 1. Montar Drive (se ainda não estiver montado)
drive.mount('/content/drive')

# 2. Configurações Globais
DATASET_DIR = "/content/drive/MyDrive/Datasets/PAMAP2/"
OUTPUT_PATH = "/content/drive/MyDrive/Datasets/PAMAP2/pamap2_v2.npz"

# IDs dos sujeitos - Separando o 108 para Validação
TRAIN_SUBJECTS = [101, 102, 103, 104, 107]
VAL_SUBJECTS = [108]
TEST_SUBJECTS = [105, 106]

# Configuração da Janela Deslizante (PAMAP2 = 100Hz)
WINDOW_SIZE = 250  # 2.5 segundos
STEP_SIZE = 125    # 50% de overlap

# Definição das Colunas (Rótulos do PAMAP2)
COLUMNS = [
    "timestamp", "activityID", "heartrate",
    "handTemperature", "handAcc16_1", "handAcc16_2", "handAcc16_3",
    "handAcc6_1", "handAcc6_2", "handAcc6_3",
    "handGyro1", "handGyro2", "handGyro3",
    "handMagne1", "handMagne2", "handMagne3",
    "handOrientation1", "handOrientation2", "handOrientation3", "handOrientation4",
    "chestTemperature", "chestAcc16_1", "chestAcc16_2", "chestAcc16_3",
    "chestAcc6_1", "chestAcc6_2", "chestAcc6_3",
    "chestGyro1", "chestGyro2", "chestGyro3",
    "chestMagne1", "chestMagne2", "chestMagne3",
    "chestOrientation1", "chestOrientation2", "chestOrientation3", "chestOrientation4",
    "ankleTemperature", "ankleAcc16_1", "ankleAcc16_2", "ankleAcc16_3",
    "ankleAcc6_1", "ankleAcc6_2", "ankleAcc6_3",
    "ankleGyro1", "ankleGyro2", "ankleGyro3",
    "ankleMagne1", "ankleMagne2", "ankleMagne3",
    "ankleOrientation1", "ankleOrientation2", "ankleOrientation3", "ankleOrientation4"
]

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
def load_and_clean_data(dataset_dir):
    print(" Carregando arquivos brutos...")
    data_collection = pd.DataFrame()

    # Iterar pelos arquivos dos sujeitos (subject101.dat a subject109.dat)
    # Nota: Vamos carregar todos que existirem na pasta
    for i in range(1, 10):
        subject_id = 100 + i
        filename = os.path.join(dataset_dir, f"subject{subject_id}.dat")

        if not os.path.exists(filename):
            print(f"    Aviso: Arquivo {filename} não encontrado. Pulando.")
            continue

        print(f"   Lendo subject{subject_id}.dat...")
        df = pd.read_table(filename, header=None, sep='\s+')
        df.columns = COLUMNS
        df["subject_id"] = subject_id # Adiciona ID do sujeito
        data_collection = pd.concat([data_collection, df], ignore_index=True)

    print("\n Limpando dados...")
    # 1. Remover colunas de orientação (ruidosas/desnecessárias)
    cols_to_drop = [c for c in data_collection.columns if 'Orientation' in c]
    df_clean = data_collection.drop(cols_to_drop, axis=1)

    # 2. Remover activityID 0 (dados de transição/sem label)
    df_clean = df_clean[df_clean["activityID"] != 0]

    # 3. Forçar numérico e interpolar dados faltantes (ex: heartrate tem NaN)
    # O PAMAP2 tem alguns NaNs nos sensores wireless. Interpolação linear resolve.
    df_clean = df_clean.apply(pd.to_numeric, errors='coerce')
    df_clean = df_clean.interpolate(method='linear', limit_direction='forward')
    df_clean = df_clean.dropna() # Remove o que não deu pra interpolar

    print(f" Dados carregados. Shape Total: {df_clean.shape}")
    return df_clean

def generate_windows(data, window_size, step_size):
    """
    Segmenta os dados em janelas deslizantes.
    """
    X_windows = []
    y_windows = []

    # Agrupa por (Sujeito, Atividade) para não misturar janelas
    # Ex: Não queremos uma janela que comece no sujeito 101 e termine no 102
    grouped = data.groupby(['subject_id', 'activityID'])

    for (subject, activity), group in grouped:
        # Garante ordem temporal
        group = group.sort_values(by='timestamp')

        # Remove colunas que não são features de sensor
        # (Timestamp, ActivityID e SubjectID não entram na rede neural)
        features = group.drop(['timestamp', 'activityID', 'subject_id'], axis=1).values
        label = activity

        # Sliding Window
        for i in range(0, len(group) - window_size + 1, step_size):
            window = features[i : i + window_size]
            X_windows.append(window)
            y_windows.append(label)

    return np.array(X_windows), np.array(y_windows)

<>:16: SyntaxWarning: invalid escape sequence '\s'
<>:16: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_758/4255115791.py:16: SyntaxWarning: invalid escape sequence '\s'
  df = pd.read_table(filename, header=None, sep='\s+')


In [ ]:
# 1. Carregar e Limpar (Usando sua função original)
df_all = load_and_clean_data(DATASET_DIR)

# 2. Encoding Global dos Labels
# Em FL, todos os clientes precisam concordar que "Caminhar" é a classe 0, por exemplo.
# Por isso, fazemos o fit do LabelEncoder no dataset inteiro antes de separar.
print("\n Codificando Labels Globalmente...")
le = LabelEncoder()
df_all['activityID'] = le.fit_transform(df_all['activityID'])
print(f" Classes detectadas: {le.classes_}")

# Identificar colunas de features
cols_ignore = ['timestamp', 'activityID', 'subject_id']
feature_cols = [c for c in df_all.columns if c not in cols_ignore]

# 3. Criação dos Clientes Federados
print("\n Particionando dados em Clientes Federados (1 Sujeito = 1 Cliente)...")
clientes_dict = {}
sujeitos_unicos = df_all['subject_id'].unique()

for subject_id in sujeitos_unicos:
    print(f" Processando Cliente: {subject_id}...")

    # Isolar os dados apenas deste usuário
    df_subject = df_all[df_all['subject_id'] == subject_id].copy()

    # Divisão Temporal (80% Treino local, 20% Teste local)
    # Importante: Como é série temporal, não usamos train_test_split aleatório.
    split_idx = int(len(df_subject) * 0.8)
    df_train = df_subject.iloc[:split_idx].copy()
    df_test = df_subject.iloc[split_idx:].copy()

    # Normalização Local (Fit apenas no treino DESTE cliente)
    # Isso simula a realidade: o celular da pessoa só conhece os próprios dados
    scaler = StandardScaler()
    df_train[feature_cols] = scaler.fit_transform(df_train[feature_cols])
    df_test[feature_cols] = scaler.transform(df_test[feature_cols])

    # Segmentação em Janelas (Usando sua função original)
    X_train, y_train = generate_windows(df_train, WINDOW_SIZE, STEP_SIZE)
    X_test, y_test = generate_windows(df_test, WINDOW_SIZE, STEP_SIZE)

    # Otimização de Memória
    X_train = X_train.astype(np.float32)
    X_test = X_test.astype(np.float32)

    # Salvar no dicionário
    clientes_dict[f"client_{subject_id}"] = {
        "X_train": X_train,
        "y_train": y_train,
        "X_test": X_test,
        "y_test": y_test
    }

    print(f"   -> Treino: {X_train.shape} | Teste: {X_test.shape}")

# 4. Salvar o Dataset Federado
OUTPUT_PATH_FED = "/content/drive/MyDrive/Datasets/PAMAP2/pamap2_federated.npz"
print("\n Salvando pacote federado...")

# np.savez_compressed aceita kwargs, então podemos desempacotar o dicionário
np.savez_compressed(OUTPUT_PATH_FED, classes=le.classes_, **clientes_dict)
print(f" Arquivo federado salvo em: {OUTPUT_PATH_FED}")

 Carregando arquivos brutos...
   Lendo subject101.dat...
   Lendo subject102.dat...
   Lendo subject103.dat...
   Lendo subject104.dat...
   Lendo subject105.dat...
   Lendo subject106.dat...
   Lendo subject107.dat...
   Lendo subject108.dat...
   Lendo subject109.dat...

 Limpando dados...
 Dados carregados. Shape Total: (1942868, 43)

 Codificando Labels Globalmente...
 Classes detectadas: [ 1  2  3  4  5  6  7 12 13 16 17 24]

 Particionando dados em Clientes Federados (1 Sujeito = 1 Cliente)...
 Processando Cliente: 101...
   -> Treino: (1586, 250, 40) | Teste: (396, 250, 40)
 Processando Cliente: 102...
   -> Treino: (1672, 250, 40) | Teste: (415, 250, 40)
 Processando Cliente: 103...
   -> Treino: (1106, 250, 40) | Teste: (275, 250, 40)
 Processando Cliente: 104...
   -> Treino: (1467, 250, 40) | Teste: (367, 250, 40)
 Processando Cliente: 105...
   -> Treino: (1729, 250, 40) | Teste: (431, 250, 40)
 Processando Cliente: 106...
   -> Treino: (1586, 250, 40) | Teste: (394, 250, 